In [20]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib import colors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import multiprocessing as mp

In [21]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [22]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/srv/ccrc/LandAP/z5218916/data/PLUMBER2/"
PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"

site_names, IGBP_types, clim_types, model_names = load_default_list()

remove_site        = get_removed_site_names()

models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

# Calculate remaining sites
set_site_names      = set(site_names)
set_remove_site     = set(remove_site)
remain_sites        = set_site_names - set_remove_site
remain_sites        = list(remain_sites)

In [23]:
model_colors ={0:'red', 1: 'darkorange',2:'orange',3:'gold',4:'yellowgreen',5:'green',6:'mediumseagreen',
               7:'lime',8:'aquamarine',9:'cyan',10:'dodgerblue',11:'blue',12:'darkolivegreen',
               13:'forestgreen',14:'lime',15:'gold', 16:'orange',17:'pink',18:'pink',19:'red',20:'deeppink',
               21:'mediumorchid',22: 'darkviolet',}

IGBP_colors  = set_IGBP_colors()
clim_colors  = set_clim_colors()

<h4 style="color:green;">Annual mean NEE</h4>  

<h5 style="color:orange;">Calc annual mean</h5>  

In [17]:
def save_annual_mean(var_name, model_in, per_LAI=False):

    var       = np.zeros(170)
    Site_name = [""] * 170    # Creates a list with 170 empty strings
    lat       = np.zeros(170)
    lon       = np.zeros(170)

    for i, site_name in enumerate(remain_sites): 
        print(site_name)
        Site_name[i]       = site_name

        PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/data/PLUMBER2/nc_files/{site_name}.nc"
        PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
        PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
        file_path          = glob.glob(PLUMBER2_flux_path+"/*"+site_name+"*.nc")

        with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
            try:
                if var_name == 'NEE':
                    if model_in == 'NoahMPv401' or model_in == 'GFDL' or model_in == 'STEMMUS-SCOPE':
                        var_tmp = f.variables[model_in + '_NEE'][:].data*(-1)
                    else:
                        var_tmp = f.variables[model_in + '_NEE'][:].data
                elif var_name == 'GPP':
                    var_tmp = f.variables[model_in + '_GPP'][:].data
            except:
                print(model_in, site_name, 'not exists')
                continue

            if per_LAI:
                if model_in in models_calc_LAI:
                    # print('in models_calc_LAI', model_in)
                    LAI = read_LAI_model(site_name, model_in, model_LAI_names[model_in])
                else:
                    # print('not in models_calc_LAI', model_in)
                    LAI = read_LAI_obs(site_name, PLUMBER2_met_path)
                tmp = np.where(LAI == 0, np.nan, var_tmp/LAI)
                var[i] = np.nanmean(tmp)*365*24*3600
            else:
                var[i] = np.nanmean(var_tmp)*365*24*3600


        with nc.Dataset(file_path[0], mode='r') as f_flux:

            lat[i] = f_flux.variables['latitude'][0,0] 
            lon[i] = f_flux.variables['longitude'][0,0] 

    #             if var_name == 'NEE':
    #                 var_qc = f_flux.variables['NEE_qc'][:,0,0]
    #             elif var_name == 'GPP':
    #                 var_qc = f_flux.variables['GPP_qc'][:,0,0]

    #             time   = nc.num2date(f_flux.variables['time'][:],f_flux.variables['time'].units,
    #                                  only_use_cftime_datetimes=False,only_use_python_datetimes=True)

    #             fig1 = plt.figure(figsize=(10, 5))
    #             plot = plt.plot(time,var_qc)
    #             plt.savefig(f'./plots/{var_name}_qc_obs_{site_name}.png',dpi=300)

    var_out              = pd.DataFrame(var, columns=[var_name])
    var_out['lat']       = lat
    var_out['lon']       = lon
    var_out['site_name'] = Site_name

    if per_LAI:
        var_out.to_csv(f'./txt/{var_name}_annual_mean_per_LAI_{model_in}.csv', index=False)
    else:
        var_out.to_csv(f'./txt/{var_name}_annual_mean_{model_in}.csv', index=False)

<h5 style="color:orange;">Calc annual mean site parallally</h5>  

In [13]:
def process_site(site_info):
    site_name, var_name, model_in, per_LAI = site_info
    Site_name = site_name
    PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
    PLUMBER2_met_path = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"
    PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
    file_path = glob.glob(PLUMBER2_flux_path + "/*" + site_name + "*.nc")

    var_tmp, lat, lon = np.nan, np.nan, np.nan  # Default values

    try:
        with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
            if var_name == 'NEE':
                if model_in in ['NoahMPv401', 'GFDL', 'STEMMUS-SCOPE']:
                    var_tmp = f.variables[model_in + '_NEE'][:].data * (-1)
                else:
                    var_tmp = f.variables[model_in + '_NEE'][:].data
            elif var_name == 'GPP':
                var_tmp = f.variables[model_in + '_GPP'][:].data
    except:
        print(model_in, site_name, 'not exists')
        return np.nan, np.nan, np.nan, site_name

    if per_LAI:
        if model_in in models_calc_LAI:
            LAI = read_LAI_model(site_name, model_in, model_LAI_names[model_in])
        else:
            LAI = read_LAI_obs(site_name, PLUMBER2_met_path)
        tmp = np.where(LAI == 0, np.nan, var_tmp / LAI)
        var_value = np.nanmean(tmp) * 365 * 24 * 3600
    else:
        var_value = np.nanmean(var_tmp) * 365 * 24 * 3600

    with nc.Dataset(file_path[0], mode='r') as f_flux:
        lat = f_flux.variables['latitude'][0, 0]
        lon = f_flux.variables['longitude'][0, 0]

    return var_value, lat, lon, site_name

def run_parallel_processing(remain_sites, var_name='NEE', model_in='ORC2_r6593', per_LAI=True):
    # Create input list for parallel processing
    site_info_list = [(site, var_name, model_in, per_LAI) for site in remain_sites]

    # Use multiprocessing to parallelize the process_site function
    with mp.Pool() as pool:
        results = pool.map(process_site, site_info_list)

    # Extract results
    var, lat, lon, Site_name = zip(*results)

    # Convert to numpy arrays
    var = np.array(var)
    lat = np.array(lat)
    lon = np.array(lon)

    # Create dataframe
    var_out = pd.DataFrame(var, columns=[var_name])
    var_out['lat'] = lat
    var_out['lon'] = lon
    var_out['site_name'] = Site_name

    # Save to CSV
    if per_LAI:
        var_out.to_csv(f'./txt/{var_name}_annual_mean_per_LAI_{model_in}.csv', index=False)
    else:
        var_out.to_csv(f'./txt/{var_name}_annual_mean_{model_in}.csv', index=False)


In [14]:
# Example call to the function
run_parallel_processing(remain_sites, var_name='NEE', model_in='ORC3_r8120', per_LAI=True)

ValueError: not enough values to unpack (expected 4, got 0)

<h5 style="color:orange;">save annual mean parallally</h5>  

In [15]:
# Define a function to generate each plot
def save_annual_mean_parallal(var_name, per_LAI=False):
    
    PLUMBER2_path_site = "/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/AU-How.nc"
    f                  = nc.Dataset(PLUMBER2_path_site, mode='r')
    model_list         = f.variables[f'{var_name}_models'][:]
    model_list         = model_list.tolist()
    model_list.append('obs')
    f.close()
    
    # Create a pool of workers (28 CPUs)
    with mp.Pool(processes=28) as pool:
        # Distribute the tasks across CPUs
        pool.starmap(save_annual_mean, [(var_name, model_in, per_LAI) for model_in in model_list])

<h5 style="color:orange;">Executing</h5>  

In [ ]:
# var_name = 'NEE'
# per_LAI  = False
# save_annual_mean_parallal(var_name, per_LAI=per_LAI)

In [ ]:
# var_name = 'NEE'
# per_LAI  = True
# save_annual_mean_parallal(var_name, per_LAI=per_LAI)

In [18]:
var_name = 'GPP'
per_LAI  = False
save_annual_mean_parallal(var_name, per_LAI=per_LAI)

In [ ]:
var_name = 'GPP'
per_LAI  = True
save_annual_mean_parallal(var_name, per_LAI=per_LAI)